In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os  
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
data = pd.read_excel('/Users/strangemax/syncthing/cirfolder/DATASCIENCE FOR HEALTH/Semester 2 /DASC512-202526/Assignment2/data/combined_testdata1.xlsx')
# Display the first few rows of the dataset
print(data.head())
# Check for missing values
print(data.isnull().sum())

   report_id                                               text  \
0       5904  The trachea is midline . The cardiomediastinal...   
1      57167  The cardiomediastinal silhouette is within nor...   
2         41  Chest x - ray 2 - 14 - 12 at 1739 : Interval t...   
3         53  Left thoracentesis with interval decrease in s...   
4        216  The cardiomediastinal silhouette is normal . T...   

                                            entities num_entities  former  
0  {'1': {'tokens': 'trachea', 'label': 'Anatomy:...           35       1  
1  {'1': {'tokens': 'cardiomediastinal', 'label':...           22       1  
2  {'1': {'tokens': 'thoracentesis', 'label': 'Ob...           48       1  
3  {'1': {'tokens': 'Left', 'label': 'Anatomy::de...           19       1  
4  {'1': {'tokens': 'cardiomediastinal', 'label':...           22       1  
report_id       0
text            0
entities        0
num_entities    0
former          0
dtype: int64


In [4]:
data.head(10)

,report_id,text,entities,num_entities,former
0,5904,The trachea is midline . The cardiomediastinal...,"{'1': {'tokens': 'trachea', 'label': 'Anatomy:...",35,1
1,57167,The cardiomediastinal silhouette is within nor...,"{'1': {'tokens': 'cardiomediastinal', 'label':...",22,1
2,41,Chest x - ray 2 - 14 - 12 at 1739 : Interval t...,"{'1': {'tokens': 'thoracentesis', 'label': 'Ob...",48,1
3,53,Left thoracentesis with interval decrease in s...,"{'1': {'tokens': 'Left', 'label': 'Anatomy::de...",19,1
4,216,The cardiomediastinal silhouette is normal . T...,"{'1': {'tokens': 'cardiomediastinal', 'label':...",22,1
5,297,"Right - sided pigtail pleural drain , left - s...","{'1': {'tokens': 'Right - sided', 'label': 'An...",27,1
6,409,Chest exam demonstrates no parenchymal opaciti...,"{'1': {'tokens': 'parenchymal', 'label': 'Anat...",18,1
7,440,The right PICC catheter has been removed . The...,"{'1': {'tokens': 'right', 'label': 'Anatomy::d...",34,1
8,613,AP portable view of the chest taken on 19 / 06...,"{'1': {'tokens': 'right - sided', 'label': 'An...",43,1
9,698,The cardiomediastinal silhouette is normal . T...,"{'1': {'tokens': 'cardiomediastinal', 'label':...",22,1


In [ ]:
# Probabilistic Target Mapping ---
# Takes discrete label indicators (0 for normal, 1 for abnormal, 2 for complex/unsure) and maps them to a continuous mathematical spectrum: 0.0, 1.0, and 0.5 respectively
label_map = {
    0: 0.0,  
    2: 0.5,  
    1: 1.0   
}

# Apply the mapping to create our continuous target variable
data['target'] = data['former'].map(label_map)

data.head(10)

In [ ]:
# Stratify Train/Test Split (80/20) ---
# We use the discrete 'former' column for stratification to guarantee 
# that the 0.0, 0.5, and 1.0 classes are perfectly proportional in both splits.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    data['text'],          # target feature is the raw text data
    data['target'],        # probabilistic target (0.0, 0.5, 1.0)
    test_size=0.20,         # 20% for testing
    random_state=42,        # fixed seed for reproducibility
    stratify=data['former'] # stratify by the original discrete labels to maintain class proportions
)

# Verify the Distribution Proportions in Train/Test Splits
print("Training Set Target Proportions")
print(y_train.value_counts(normalize=True)) # Show the proportions of 0.0, 0.5, and 1.0 in the training set
print(f"Total Training Samples: {len(y_train)}\n") # Show the total number of samples in the training set

print("Testing Set Target Proportions") # Show the proportions of 0.0, 0.5, and 1.0 in the testing set
print(y_test.value_counts(normalize=True)) # Show the total number of samples in the testing set
print(f"Total Testing Samples: {len(y_test)}") # Show the total number of samples in the testing set
# proportions confirmed to be consistent across both sets, ensuring a representative distribution for model training and evaluation.

Training Set Target Proportions
target
1.0    0.815126
0.5    0.142857
0.0    0.042017
Name: proportion, dtype: float64
Total Training Samples: 119

Testing Set Target Proportions
target
1.0    0.833333
0.5    0.133333
0.0    0.033333
Name: proportion, dtype: float64
Total Testing Samples: 30


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# take raw text reports and soft labels and processes them into standard structured tensors 
# that a Deep Learning Neural Network can process in mini-batches
# We use 'emilyalsentzer/Bio_ClinicalBERT' because it was pre-trained on MIMIC-III clinical notes.
# standard BERT does not understand like thoracentesis because they are outside normal english vocabulary
print("Loading ClinicalBERT Tokenizer...GET EXITED!!!")
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# CREATE A CUSTOM PYTORCH DATASET
class RadiologyDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len=256):
        self.texts = texts.values if hasattr(texts, 'values') else list(texts)
        self.targets = targets.values if hasattr(targets, 'values') else list(targets)
        self.tokenizer = tokenizer
        self.max_len = max_len # pads or truncates to this length for uniform input size
        
    def __len__(self): # Tells PyTorch exactly how many patient records are in the dataset
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        target = self.targets[idx]
        
        # Tokenize the text sequence, pad or truncate to a fixed max length
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True, # 1 for r# eal clinical words, 0 for pad tokens
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'target': torch.tensor(target, dtype=torch.float) # Kept as float for probabilistic regression
        }

# INSTANTIATE TRAINING AND TESTING DATASETS
# using the X_train_raw, X_test_raw, y_train, y_test from your original stratified split using the same seed in shallow model
train_dataset = RadiologyDataset(X_train_raw, y_train, tokenizer)
test_dataset = RadiologyDataset(X_test_raw, y_test, tokenizer)

# Create DataLoaders for batching
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"PyTorch Datasets Ready.")
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Loading ClinicalBERT Tokenizer...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

PyTorch Datasets Ready.
Train batches: 15 | Test batches: 4
